In [2]:
# ============================================================
# EDUVISION_DV - MODULE 3: EDUCATION KPI ENGINEERING
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path


# ------------------------------------------------------------
# 1. Define project paths
# ------------------------------------------------------------

BASE_DIR = Path(__file__).resolve().parent.parent

INPUT_FILE = (
    BASE_DIR
    / "data"
    / "processed"
    / "university_cleaned.csv"
)

OUTPUT_FILE = (
    BASE_DIR
    / "data"
    / "processed"
    / "university_final_dataset.xlsx"
)


# ------------------------------------------------------------
# 2. Load Module 2 cleaned dataset
# ------------------------------------------------------------

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Input file not found:\n{INPUT_FILE}"
    )

df = pd.read_csv(INPUT_FILE)

print("=" * 70)
print("EDUVISION_DV - MODULE 3: KPI ENGINEERING")
print("=" * 70)

print(f"\nInput rows    : {len(df):,}")
print(f"Input columns : {len(df.columns)}")


# ------------------------------------------------------------
# 3. Calculate KPI 1 - Global Ranking Score
# ------------------------------------------------------------
# Average of QS and THE normalized ranking scores.
# Both source values are already on a 0-100 scale.
# ------------------------------------------------------------

df["Global_Ranking_Score"] = (
    df[["QS_Normalized_Rank", "THE_Normalized_Rank"]]
    .mean(axis=1, skipna=True)
)


# ------------------------------------------------------------
# 4. Calculate KPI 2 - Research Impact Score
# ------------------------------------------------------------
# Equal-weight average of:
#   QS Citations per Faculty
#   THE Research Quality
# ------------------------------------------------------------

df["Research_Impact_Score"] = (
    df[["QS_Citations_Per_Faculty", "THE_Research_Quality"]]
    .mean(axis=1, skipna=True)
)


# ------------------------------------------------------------
# 5. Calculate KPI 3 - Faculty-to-Student Ratio
# ------------------------------------------------------------
# THE provides Student-to-Staff Ratio.
#
# Conversion:
# Faculty per 100 students = 100 / Student-to-Staff Ratio
# ------------------------------------------------------------

df["Faculty_to_Student_Ratio"] = np.where(
    df["THE_Student_Staff_Ratio"] > 0,
    100 / df["THE_Student_Staff_Ratio"],
    np.nan
)


# ------------------------------------------------------------
# 6. Calculate KPI 4 - International Student Percentage
# ------------------------------------------------------------
# THE already provides the percentage value.
# ------------------------------------------------------------

df["International_Student_Percentage"] = (
    df["THE_International_Students"]
)


# ------------------------------------------------------------
# 7. Calculate KPI 5 - Academic Reputation Score
# ------------------------------------------------------------
# QS provides Academic Reputation Score directly.
# ------------------------------------------------------------

df["Academic_Reputation_Score"] = (
    df["QS_Academic_Reputation"]
)


# ------------------------------------------------------------
# 8. Calculate KPI 6 - Research Productivity Index
# ------------------------------------------------------------
# Equal-weight average of:
#   THE Research Environment
#   THE Research Quality
# ------------------------------------------------------------

df["Research_Productivity_Index"] = (
    df[["THE_Research_Environment", "THE_Research_Quality"]]
    .mean(axis=1, skipna=True)
)


# ------------------------------------------------------------
# 9. Define KPI columns
# ------------------------------------------------------------

kpi_columns = [
    "Global_Ranking_Score",
    "Research_Impact_Score",
    "Faculty_to_Student_Ratio",
    "International_Student_Percentage",
    "Academic_Reputation_Score",
    "Research_Productivity_Index"
]


# ------------------------------------------------------------
# 10. Validate KPI ranges
# ------------------------------------------------------------

score_kpis = [
    "Global_Ranking_Score",
    "Research_Impact_Score",
    "International_Student_Percentage",
    "Academic_Reputation_Score",
    "Research_Productivity_Index"
]

for column in score_kpis:

    invalid_count = (
        (df[column] < 0) |
        (df[column] > 100)
    ).sum()

    if invalid_count > 0:
        raise ValueError(
            f"{column} contains {invalid_count} invalid values."
        )


invalid_ratio = (
    df["Faculty_to_Student_Ratio"] <= 0
).sum()

if invalid_ratio > 0:
    raise ValueError(
        "Faculty_to_Student_Ratio contains invalid values."
    )


# ------------------------------------------------------------
# 11. Check duplicate universities
# ------------------------------------------------------------

duplicate_universities = df["University"].duplicated().sum()

if duplicate_universities > 0:
    raise ValueError(
        f"Found {duplicate_universities} duplicate universities."
    )


# ------------------------------------------------------------
# 12. Arrange final columns
# ------------------------------------------------------------

final_columns = [

    "University",
    "Country",

    # QS fields
    "QS_Rank",
    "QS_Overall_Score",
    "QS_Academic_Reputation",
    "QS_Employer_Reputation",
    "QS_Faculty_Student",
    "QS_Citations_Per_Faculty",
    "QS_International_Faculty",
    "QS_International_Students",
    "QS_International_Research_Network",
    "QS_Employment_Outcomes",
    "QS_Sustainability",
    "QS_Normalized_Rank",

    # THE fields
    "THE_Rank",
    "THE_Overall_Score",
    "THE_Teaching",
    "THE_Research_Environment",
    "THE_Research_Quality",
    "THE_Industry_Impact",
    "THE_International_Outlook",
    "THE_Number_Students",
    "THE_Student_Staff_Ratio",
    "THE_International_Students",
    "THE_Female_Male_Ratio",
    "THE_Year",
    "THE_Normalized_Rank",
    "THE_Female_Percentage",
    "THE_Male_Percentage",

    # KPI fields
    "Global_Ranking_Score",
    "Research_Impact_Score",
    "Faculty_to_Student_Ratio",
    "International_Student_Percentage",
    "Academic_Reputation_Score",
    "Research_Productivity_Index"
]

final_df = df[final_columns].copy()


# ------------------------------------------------------------
# 13. Save final Excel dataset
# ------------------------------------------------------------

final_df.to_excel(
    OUTPUT_FILE,
    index=False,
    sheet_name="University_Data"
)


# ------------------------------------------------------------
# 14. Completion message
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MODULE 3 COMPLETED SUCCESSFULLY")
print("=" * 70)

print(f"\nFinal rows    : {len(final_df):,}")
print(f"Final columns : {len(final_df.columns)}")
print(f"\nOutput file   : {OUTPUT_FILE}")

print("\nKPIs generated:")

for kpi in kpi_columns:
    print(f"  ✓ {kpi}")

NameError: name '__file__' is not defined